# reBAP cost of our picks vs. SMARD

Prices the hourly forecast errors of spec 09's picks ([`visualization-01-regression-best-models.ipynb`](visualization-01-regression-best-models.ipynb)) and of SMARD at the quarter-hourly reBAP; models, method and caveats are in [`regression-models-claude.ipynb`](regression-models-claude.ipynb).
Reads `data/rebap.csv`, `data/models/{model,ensemble}_{forecast_errors_hourly,scoreboard}.csv` and `data/metrics/smard_forecast_errors_hourly.csv`; writes `data/models/model_rebap_cost_hourly.csv`.
*The money is a proxy: national forecast error valued at the imbalance price, not a settlement.*

## 1 Setup and checks

### 1.1 Settings

In [ ]:
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap, to_hex
from matplotlib.lines import Line2D
from matplotlib.patches import FancyBboxPatch, Patch
from matplotlib.transforms import ScaledTranslation

PICKS = {  # by hand: spec 09 rank 1; an ensemble key only if it beats that pick (spec 10 rule)
    "overall": "ensemble_regime_weighted",
    "below_zero": "ensemble_regime_weighted",
    "low_extreme": "random_forest_hybrid",
    "high_extreme": "xgb_hybrid",
}
ENSEMBLE_PICK = "ensemble_regime_weighted"  # spec 09's ensemble pick, a money-table row
SPLIT = "rolling"               # spec 09's CANDIDATE_SPLIT
MIN_BIN_SHARE = 0.5             # spec 09's forecast-share floor, used by the pick check
PRICE_BANDS = [0.5, 0.9, 0.99]  # quantiles of the hourly |reBAP| over the common hours
TOP_HOURS = 10                  # hours in the top-hours plot
EXPORT_ENABLED = True

# Label and colour per row, copied from visualization-01-regression-best-models.ipynb
STYLE = {
    "sarimax_fourier": {"label": "SARIMAX + Fourier", "color": "#D9A53A"},
    "lgbm_direct": {"label": "LightGBM direct", "color": "#2C6EBA"},
    "lgbm_hybrid": {"label": "LightGBM hybrid", "color": "#2F8F5B"},
    "xgb_direct": {"label": "XGBoost direct", "color": "#E95D0F"},
    "xgb_hybrid": {"label": "XGBoost hybrid", "color": "#B10F0F"},
    "linear_direct": {"label": "Ridge direct", "color": "#7A4FA3"},
    "random_forest_hybrid": {"label": "Random forest hybrid", "color": "#8C564B"},
    # Ensembles: labels from METHODS in ensemble-claude.ipynb, one colour (only one is shown)
    "ensemble_equal_mean": {"label": "Ensemble: equal mean", "color": "#C2387A"},
    "ensemble_weighted": {"label": "Ensemble: weighted", "color": "#C2387A"},
    "ensemble_regime_weighted": {"label": "Ensemble: regime-weighted", "color": "#C2387A"},
    "ensemble_adaptive": {"label": "Ensemble: adaptive", "color": "#C2387A"},
    "actual": {"label": "Actual residual load", "color": "#1C1C1C"},
    "smard": {"label": "SMARD day-ahead", "color": "#48505A", "linestyle": "--"},
}

# Bin regions and edges on a residual-load axis, never a model's colour
TAIL_COLOR = {
    "low_extreme": "#17BECF",   # cyan
    "below_zero": "#9EDAE5",    # light cyan
    "high_extreme": "#E6B800",  # gold
    "ordinary": "#E3E8EF",      # light grey
}

# Price bands, cheapest to dearest: a slate-indigo ramp, one colour per band
PRICE_RAMP = LinearSegmentedColormap.from_list("price", ["#E6E6F2", "#3B3B6E"])
PRICE_COLOR = [to_hex(PRICE_RAMP(x)) for x in np.linspace(0, 1, len(PRICE_BANDS) + 1)]
taken = {spec["color"].lower() for spec in STYLE.values()} | {c.lower() for c in TAIL_COLOR.values()}
assert not taken & set(PRICE_COLOR), "PRICE_COLOR repeats a STYLE or TAIL_COLOR colour"


def label_of(row):
    """Display label of a (model, split_method) row."""
    return STYLE[row[0]]["label"]


def style_timeseries(ax, title, ylabel):
    """Apply the EDA time-series chart style to the matplotlib axes.

    `ylabel` is required: every plot must state its unit.
    """
    ax.set_title(title, loc="center", fontsize=15, pad=12)
    ax.set_xlabel("")
    ax.set_ylabel(ylabel, color="grey")

    # Light horizontal grid lines behind the data, no box on the top and right
    ax.grid(axis="y", color="0.9", linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)

    # Tick labels without tick marks
    ax.tick_params(colors="black", length=0)

    # x axis: one label per year, minor ticks every quarter
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_minor_locator(mdates.MonthLocator((1, 4, 7, 10)))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    # y axis: thousands separator, no decimals (e.g. 45,000)
    ax.yaxis.set_major_formatter(lambda value, _: f"{value:,.0f}")


def style_plain(ax, title, ylabel, xlabel):
    """The same look for plots without a time axis."""
    ax.set_title(title, loc="center", fontsize=15, pad=12)
    ax.set_xlabel(xlabel, color="grey")
    ax.set_ylabel(ylabel, color="grey")
    ax.grid(axis="y", color="0.9", linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.tick_params(colors="black", length=0)
    ax.yaxis.set_major_formatter(lambda value, _: f"{value:,.0f}")

### 1.2 Loading

In [ ]:
# data/: first in the working directory, then in each parent
DATA_DIR = next(
    (folder / "data" for folder in [Path.cwd(), *Path.cwd().parents] if (folder / "data").is_dir()),
    None,
)
if DATA_DIR is None:
    raise RuntimeError(f"No data/ folder found in {Path.cwd()} or any parent folder.")

MODEL_SOURCE = "notebooks/05_modeling/regression-models-claude.ipynb with EXPORT_ENABLED = True"
ENSEMBLE_SOURCE = "notebooks/05_modeling/ensemble-claude.ipynb with EXPORT_ENABLED = True"
SMARD_SOURCE = "notebooks/02_forecast_metrics/forecast-metrics-claude.ipynb"
REBAP_SOURCE = "notebooks/API-connection.ipynb Part 2 (needs .env)"
SOURCES = {
    "scoreboard": ("models/model_scoreboard.csv", MODEL_SOURCE),
    "hourly": ("models/model_forecast_errors_hourly.csv", MODEL_SOURCE),
    "ensemble_scoreboard": ("models/ensemble_scoreboard.csv", ENSEMBLE_SOURCE),
    "ensemble_hourly": ("models/ensemble_forecast_errors_hourly.csv", ENSEMBLE_SOURCE),
    "smard": ("metrics/smard_forecast_errors_hourly.csv", SMARD_SOURCE),
    "rebap": ("rebap.csv", REBAP_SOURCE),
}


def path_of(name):
    """Path of one input file; stops if it is missing."""
    path, producer = SOURCES[name]
    path = DATA_DIR / path
    if not path.exists():
        raise FileNotFoundError(f"{path} not found. Create it by running {producer}.")
    return path


def load(name):
    """One export as a DataFrame."""
    frame = pd.read_csv(path_of(name))
    if "timestamp" in frame.columns:
        frame["timestamp"] = pd.to_datetime(frame["timestamp"], format="%Y-%m-%d %H:%M:%S")
    return frame


scoreboard = load("scoreboard")
hourly = load("hourly")
smard = load("smard").set_index("timestamp")

# Ensemble exports: their model rows must equal the model export, then add the ensembles
ensemble_scoreboard = load("ensemble_scoreboard")
shared = scoreboard.merge(
    ensemble_scoreboard, on=["model", "split_method", "table", "metric"], suffixes=("", "_ens")
)
differs = ~np.isclose(shared["value"], shared["value_ens"], equal_nan=True)
differs |= shared["count"] != shared["count_ens"]
if differs.any():
    stale = sorted(set(shared.loc[differs, "model"]))
    raise RuntimeError(
        f"{stale} differ between the ensemble and the model exports. "
        f"Re-run {ENSEMBLE_SOURCE} after {MODEL_SOURCE}."
    )
is_ensemble = ensemble_scoreboard["model"].str.startswith("ensemble_")
scoreboard = pd.concat([scoreboard, ensemble_scoreboard[is_ensemble]], ignore_index=True)
hourly = pd.concat([hourly, load("ensemble_hourly")], ignore_index=True)

# reBAP: German CSV, one row per quarter-hour, timestamp = local interval start
rebap = pd.read_csv(path_of("rebap"), delimiter=";", encoding="utf-8-sig")
rebap["timestamp"] = pd.to_datetime(rebap["timestamp"], format="%Y-%m-%d %H:%M")
PRICE_COLUMNS = ["reBAP unterdeckt", "reBAP ueberdeckt"]
for column in PRICE_COLUMNS:
    rebap[column] = rebap[column].astype(str).str.replace(",", ".").astype(float)

SMARD_ROW = ("smard", "none")
TEST_HOURS = pd.DatetimeIndex(hourly["timestamp"].unique()).sort_values()
TEST_DAYS = TEST_HOURS.normalize().unique()
ACTUAL = hourly.groupby("timestamp")["residual_load"].first().reindex(TEST_HOURS)

# Rows: SMARD, every SPLIT registry row (spec 09's candidates), ENSEMBLE_PICK and the picks
registry = sorted(
    {
        row
        for row in zip(hourly["model"], hourly["split_method"])
        if row[1] == SPLIT and row[0] not in ("smard", "seasonal_naive") and not row[0].startswith("ensemble_")
    }
)
MODEL_ROWS = list(dict.fromkeys([*registry, (ENSEMBLE_PICK, SPLIT), *((model, SPLIT) for model in PICKS.values())]))
missing_style = sorted({model for model, _ in MODEL_ROWS} - STYLE.keys())
if missing_style:
    raise KeyError(
        f"No STYLE entry for {missing_style}. "
        "Copy its label and colour from MODELS in regression-models-claude.ipynb "
        "(ensembles: METHODS in ensemble-claude.ipynb)."
    )

# One column per exported (model, split_method) row
FORECAST = hourly.pivot(index="timestamp", columns=["model", "split_method"], values="forecast")
ERROR = hourly.pivot(index="timestamp", columns=["model", "split_method"], values="err_residual_load")

# Common hours (spec 09): an actual, SMARD's forecast and a forecast from every exported row,
# seasonal naive included, so the hours match the scoreboard
has_all = ACTUAL.notna() & smard["fc_residual_load"].reindex(TEST_HOURS).notna()
has_all &= FORECAST.reindex(TEST_HOURS).notna().all(axis=1)
COMMON = TEST_HOURS[has_all.to_numpy()]

# Money rows only; a missing row stays empty and fails the pick check
FORECAST = FORECAST.reindex(index=TEST_HOURS, columns=pd.MultiIndex.from_tuples(MODEL_ROWS))
ERROR = ERROR.reindex(index=TEST_HOURS, columns=FORECAST.columns)

# Error per row on the common hours (forecast − actual), SMARD first
ERRORS = pd.concat(
    {SMARD_ROW: smard.loc[COMMON, "err_residual_load"], **{row: ERROR.loc[COMMON, row] for row in MODEL_ROWS}},
    axis=1,
)

print(f"test window : {TEST_DAYS[0]:%Y-%m-%d} .. {TEST_DAYS[-1]:%Y-%m-%d} ({len(TEST_HOURS):,} hours)")
print(f"picks       : " + ", ".join(f"{category} {model}" for category, model in PICKS.items()) + f" ({SPLIT} split)")
print(f"money rows  : {len(MODEL_ROWS)} ({', '.join(model for model, _ in MODEL_ROWS)})")
print(f"common hours: {len(COMMON):,} of {len(TEST_HOURS):,}")
print(f"reBAP file  : {rebap['timestamp'].min():%Y-%m-%d %H:%M} .. {rebap['timestamp'].max():%Y-%m-%d %H:%M} ({len(rebap):,} quarter-hours)")

### 1.3 Pick check

Every pick and `ENSEMBLE_PICK` needs a `SPLIT` row; each pick must pass spec 09's rule in its category. On a failure the notebook stops; after spec 09 is re-run with new picks, edit `PICKS`.

In [ ]:
CATEGORIES = ["overall", "below_zero", "low_extreme", "high_extreme"]
TAILS = CATEGORIES[1:]
VALUE = scoreboard.pivot(index=["model", "split_method"], columns="metric", values="value")
COUNT = scoreboard.pivot(index=["model", "split_method"], columns="metric", values="count")
HOURLY_ROWS = set(zip(hourly["model"], hourly["split_method"]))


def forecast_share(row, name):
    """By-forecast hours / by-actual hours of a row in a bin."""
    by_actual = COUNT.loc[row, f"MAE_{name}_by_actual"]
    return COUNT.loc[row, f"MAE_{name}_by_forecast"] / by_actual if by_actual else np.nan


def failed_rules(row, category):
    """Spec 09's pick rule: the rules a row fails in a category; empty if it qualifies."""
    if category == "overall":
        skill = VALUE.loc[row, "skill_pct"]
        return [] if skill > 0 else [f"skill vs SMARD {skill:+.1f} % not above 0"]
    failed = []
    for view in ["actual", "forecast"]:
        metric = f"MAE_{category}_by_{view}"
        if not VALUE.loc[row, metric] < VALUE.loc[SMARD_ROW, metric]:
            failed.append(f"{metric} {VALUE.loc[row, metric]:,.0f} not below SMARD's {VALUE.loc[SMARD_ROW, metric]:,.0f}")
    share = forecast_share(row, category)
    if not share >= MIN_BIN_SHARE:
        failed.append(f"forecast share {share:.2f} < {MIN_BIN_SHARE}")
    return failed


problems = []
for key, model in [*PICKS.items(), ("ENSEMBLE_PICK", ENSEMBLE_PICK)]:
    row = (model, SPLIT)
    if row not in HOURLY_ROWS:
        problems.append(f"{key} {model}: no {SPLIT} row in the hourly exports")
    if row not in VALUE.index:
        problems.append(f"{key} {model}: no {SPLIT} row in the scoreboards")
        continue
    if key == "ENSEMBLE_PICK":
        continue  # a money-table row; it need not qualify
    failed = failed_rules(row, key)
    problems += [f"{key} {model}: {rule}" for rule in failed]
    if key == "overall":
        detail = f"skill vs SMARD {VALUE.loc[row, 'skill_pct']:+.1f} %"
    else:
        by_actual, by_forecast = (f"MAE_{key}_by_{view}" for view in ["actual", "forecast"])
        detail = (
            f"MAE by actual {VALUE.loc[row, by_actual]:,.0f} (SMARD {VALUE.loc[SMARD_ROW, by_actual]:,.0f}), "
            f"by forecast {VALUE.loc[row, by_forecast]:,.0f} (SMARD {VALUE.loc[SMARD_ROW, by_forecast]:,.0f}), "
            f"forecast share {forecast_share(row, key):.2f}"
        )
    print(f"{key:<12} {model:<24} {detail}: {'fails' if failed else 'passes'}")

if problems:
    raise RuntimeError(
        f"pick check failed for PICKS = {PICKS}, ENSEMBLE_PICK = {ENSEMBLE_PICK!r}:\n  " + "\n  ".join(problems)
        + "\nRe-run visualization-01-regression-best-models.ipynb and edit PICKS; no model is picked here."
    )
print("pick check passed")

### 1.4 Hourly price and cost

On an autumn switch day `rebap.csv` lists 02:00–02:45 twice (CEST, then CET); SMARD's single 02:00 row is the CET hour, so the last occurrence is kept. Each hour's price is the mean of its four `|reBAP|` quarter-hours; `cost = |error| × price`, `saved = SMARD's cost − the row's cost` (positive: the row is cheaper).

In [ ]:
# Fold rule: keep the second (CET) occurrence of a repeated local quarter-hour
folded = rebap["timestamp"].duplicated(keep=False)
quarters = rebap.drop_duplicates("timestamp", keep="last").copy()
quarters["hour"] = quarters["timestamp"].dt.floor("h")  # SMARD's naive local key
quarters = quarters[quarters["hour"].isin(COMMON)]

# Hourly price: mean |reBAP| prices every cost, the signed mean goes into the export only
by_hour = quarters.groupby("hour")["reBAP unterdeckt"]
PRICE_ABS = by_hour.apply(lambda price: price.abs().mean()).reindex(COMMON)
PRICE = by_hour.mean().reindex(COMMON)

# Cost per row (EUR) and saving against SMARD (EUR, positive: the row is cheaper)
COST = ERRORS.abs().mul(PRICE_ABS, axis=0)
SAVED = COST.drop(columns=SMARD_ROW).rsub(COST[SMARD_ROW], axis=0)

# Bins by the actual; edges are quantiles of the actual over the test hours (spec 09)
P1, P99 = ACTUAL.quantile([0.01, 0.99]).to_numpy()
ZERO = 0.0  # MWh: below_zero bin
BIN_RULE = {"below_zero": "< 0 MWh", "low_extreme": f"≤ P1 ({P1:,.0f} MWh)", "high_extreme": f"> P99 ({P99:,.0f} MWh)"}
actual = ACTUAL[COMMON]
IN_BIN = pd.DataFrame(
    {"below_zero": actual < ZERO, "low_extreme": actual <= P1, "high_extreme": actual > P99}, index=COMMON
)
HOURS_OF = {"overall": pd.Series(True, index=COMMON), **{name: IN_BIN[name] for name in TAILS}}

fold_hours = sorted(set(rebap.loc[folded, "timestamp"].dt.floor("h")) & set(COMMON))
print(f"fold rule   : {len(fold_hours)} autumn hour(s) in the window priced with the CET quarters: "
      + ", ".join(f"{hour:%Y-%m-%d %H:%M}" for hour in fold_hours))
print(f"price       : mean |reBAP| {PRICE_ABS.mean():,.0f} EUR/MWh, max {PRICE_ABS.max():,.0f} EUR/MWh")
print(f"SMARD cost  : {COST[SMARD_ROW].sum() / 1e6:,.1f} M€ over {len(COMMON):,} hours")

### 1.5 Self-check

In [ ]:
REGENERATE_MODELS = f"data/models/ ({MODEL_SOURCE}, then {ENSEMBLE_SOURCE})"
REGENERATE_ALL = f"data/metrics/ ({SMARD_SOURCE}), then {REGENERATE_MODELS}"
REGENERATE_REBAP = f"data/rebap.csv (re-run {REBAP_SOURCE})"


def require(ok, problem, regenerate):
    """Stop the notebook, naming the files to regenerate."""
    if not ok:
        raise RuntimeError(f"self-check failed: {problem}. Regenerate {regenerate}.")


require(len(COMMON) > 0, "no common hours", REGENERATE_ALL)

# Snapshot: same actuals in the model export and the SMARD file
actual_gap = float((ACTUAL[COMMON] - smard.loc[COMMON, "residual_load"]).abs().max())
require(
    actual_gap <= 1e-6,
    f"residual_load differs between the two hourly files by up to {actual_gap:,.2f} MWh",
    REGENERATE_ALL,
)

# Scoreboard: rebuilt MAE and hour count of every pick and of SMARD
for row in [SMARD_ROW, *dict.fromkeys((model, SPLIT) for model in PICKS.values())]:
    error = ERRORS[row]
    mae, hours = error.abs().mean(), len(error)
    exported_mae, exported_hours = VALUE.loc[row, "MAE"], COUNT.loc[row, "MAE"]
    require(
        abs(mae - exported_mae) <= 1e-6 and hours == exported_hours,
        f"{row}: MAE {mae:,.3f} on {hours:,} h, scoreboard {exported_mae:,.3f} on {exported_hours:,.0f} h",
        REGENERATE_ALL if row == SMARD_ROW else REGENERATE_MODELS,
    )

# reBAP file: one price for both directions, in EUR/MWh
unequal = int((rebap["reBAP unterdeckt"] != rebap["reBAP ueberdeckt"]).sum())
require(unequal == 0, f"reBAP unterdeckt and ueberdeckt differ on {unequal:,} rows", REGENERATE_REBAP)
units = sorted(rebap["Einheit"].unique())
require(units == ["EUR/MWh"], f"Einheit is {units}, not EUR/MWh", REGENERATE_REBAP)
datentyp = quarters["Datentyp"].value_counts()
print("Datentyp in the window: " + ", ".join(f"{kind} {count:,}" for kind, count in datentyp.items()))

# Coverage: after the fold rule, four quarter-hours in every common hour
per_hour = quarters.groupby("hour").size().reindex(COMMON, fill_value=0)
short = per_hour[per_hour != 4]
require(
    short.empty,
    f"{len(short):,} common hours without exactly 4 reBAP quarter-hours "
    f"(first {short.index[0]:%Y-%m-%d %H:%M}: {short.iloc[0]})" if len(short) else "",
    REGENERATE_REBAP,
)
require(PRICE_ABS.notna().all(), "a common hour has no price", REGENERATE_REBAP)

print(
    f"self-check passed: one snapshot, every common hour priced "
    f"({len(COMMON):,} hours, {len(quarters):,} quarter-hours)"
)

## 2 Overview

### 2.1 Edges and hours

Bins by the actual (spec 09's edges over the test hours); price bands by the hourly `|reBAP|` over the common hours, one set for every category.

In [ ]:
# Price bands: PRICE_BANDS quantiles of the hourly |reBAP|, cheapest band first
BAND_EDGES = PRICE_ABS.quantile(PRICE_BANDS).to_numpy()
levels = [0, *(round(100 * level, 6) for level in PRICE_BANDS), 100]
BAND_NAMES = (
    [f"bottom {levels[1]:g} %"]
    + [f"{low:g}–{high:g} %" for low, high in zip(levels[1:-2], levels[2:-1])]
    + [f"top {100 - levels[-2]:g} %"]
)
BAND = pd.Series(np.searchsorted(BAND_EDGES, PRICE_ABS.to_numpy(), side="left"), index=COMMON)  # 0 = cheapest
TOP_BAND = BAND == len(BAND_NAMES) - 1
BAND_RANGE = [
    f"≤ {BAND_EDGES[0]:,.0f}",
    *(f"{low:,.0f}–{high:,.0f}" for low, high in zip(BAND_EDGES[:-1], BAND_EDGES[1:])),
    f"> {BAND_EDGES[-1]:,.0f}",
]

print(f"bin edges  : P1 {P1:,.0f} MWh, zero {ZERO:,.0f} MWh, P99 {P99:,.0f} MWh (actual over {len(TEST_HOURS):,} test hours)")
print(f"price bands: " + ", ".join(f"{name} {span}" for name, span in zip(BAND_NAMES, BAND_RANGE))
      + f" EUR/MWh (|reBAP| over {len(COMMON):,} common hours)")
print()
print(f"{'category':<13} {'rule':<26} {'hours':>6} {'mean |reBAP|':>13}   hours per price band")
for category in CATEGORIES:
    in_category = HOURS_OF[category]
    rule = "all common hours" if category == "overall" else f"actual {BIN_RULE[category]}"
    per_band = BAND[in_category].value_counts().reindex(range(len(BAND_NAMES)), fill_value=0)
    mean_price = f"{PRICE_ABS[in_category].mean():,.0f}" if in_category.any() else "–"
    print(f"{category:<13} {rule:<26} {in_category.sum():>6,} {mean_price:>13}   "
          + " / ".join(f"{count:,}" for count in per_band))

### 2.2 Picks table

One row per category: its pick against SMARD on the category's hours. `top-band share` is the share of `saved` that comes from hours in the top price band.

In [ ]:
def money(row, category):
    """Cost, saving and MWh skill of one row against SMARD on a category's hours."""
    in_category = HOURS_OF[category]
    smard_cost, row_cost = COST.loc[in_category, SMARD_ROW].sum(), COST.loc[in_category, row].sum()
    saved = SAVED.loc[in_category, row].sum()
    smard_mwh, row_mwh = ERRORS.loc[in_category, SMARD_ROW].abs().sum(), ERRORS.loc[in_category, row].abs().sum()
    if not in_category.any():
        return {"hours": 0}
    return {
        "hours": int(in_category.sum()),
        "SMARD cost": smard_cost,
        "cost": row_cost,
        "saved": saved,
        "saved %": 100 * saved / smard_cost,
        "MWh skill %": 100 * (1 - row_mwh / smard_mwh),
        "top-band share %": 100 * SAVED.loc[in_category & TOP_BAND, row].sum() / saved,
    }


records = []
for category, model in PICKS.items():
    figures = money((model, SPLIT), category)
    records.append(
        {
            "category": category,
            "pick": label_of((model, SPLIT)),
            "hours": figures["hours"],
            "mean |reBAP| (EUR/MWh)": PRICE_ABS[HOURS_OF[category]].mean(),
            "SMARD cost (M€)": figures.get("SMARD cost", np.nan) / 1e6,
            "pick cost (M€)": figures.get("cost", np.nan) / 1e6,
            "saved (M€)": figures.get("saved", np.nan) / 1e6,
            "saved %": figures.get("saved %"),
            "MWh skill %": figures.get("MWh skill %"),
            "top-band share %": figures.get("top-band share %"),
        }
    )
PICKS_TABLE = pd.DataFrame(records).set_index("category")

print(
    f"Picks vs SMARD, {TEST_DAYS[0]:%Y-%m-%d} .. {TEST_DAYS[-1]:%Y-%m-%d}, {len(COMMON):,} common hours. "
    "The money is a proxy: national forecast error valued at the imbalance price, not a settlement."
)
display(
    PICKS_TABLE.style.format(
        {
            "hours": "{:,}",
            "mean |reBAP| (EUR/MWh)": "{:,.0f}",
            "SMARD cost (M€)": "{:,.1f}",
            "pick cost (M€)": "{:,.1f}",
            "saved (M€)": "{:+,.1f}",
            "saved %": "{:+.1f}",
            "MWh skill %": "{:+.1f}",
            "top-band share %": "{:.0f}",
        },
        na_rep="–",
    )
)

### 2.3 Money table

Every money-table row in every category. `money rank`: by `saved`; `MAE rank`: spec 09's key (months beating SMARD, then MAE, for `overall`; MAE by actual for the bins) among the rows that pass its rule, `–` for the others. The category's pick is in bold; the table never changes `PICKS`.

In [ ]:
def rank_key(row, category):
    """Spec 09's sort key: best first, ties by model key."""
    if category == "overall":
        return (-VALUE.loc[row, "months_beating_smard"], VALUE.loc[row, "MAE"], row[0])
    return (VALUE.loc[row, f"MAE_{category}_by_actual"], row[0])


groups = {}
for category in CATEGORIES:
    saved = pd.Series({row: SAVED.loc[HOURS_OF[category], row].sum() for row in MODEL_ROWS})
    qualified = sorted((row for row in MODEL_ROWS if not failed_rules(row, category)), key=lambda row: rank_key(row, category))
    mae_rank = {row: rank for rank, row in enumerate(qualified, start=1)}
    groups[category] = pd.DataFrame(
        {
            "saved (M€)": saved / 1e6,
            "saved %": 100 * saved / COST.loc[HOURS_OF[category], SMARD_ROW].sum(),
            "money rank": saved.rank(ascending=False, method="min").astype(int).astype(str),
            "MAE rank": [str(mae_rank[row]) if row in mae_rank else "–" for row in MODEL_ROWS],
        },
        index=MODEL_ROWS,
    )
MONEY_TABLE = pd.concat(groups, axis=1)
MONEY_TABLE = MONEY_TABLE.loc[sorted(MODEL_ROWS, key=lambda row: VALUE.loc[row, "MAE"])]
MONEY_TABLE.index = [label_of(row) for row in MONEY_TABLE.index]


def bold_pick(column):
    """Bold the pick's cell in its category's columns."""
    pick = label_of((PICKS[column.name[0]], SPLIT))
    return ["font-weight: bold" if label == pick else "" for label in column.index]


print(f"Money table: {len(MODEL_ROWS)} {SPLIT} rows, saved against SMARD per category (rows by test MAE; bold: the pick)")
display(
    MONEY_TABLE.style.format(
        {(category, "saved (M€)"): "{:+,.1f}" for category in CATEGORIES}
        | {(category, "saved %"): "{:+.1f}" for category in CATEGORIES},
        na_rep="–",
    ).apply(bold_pick)
)

## 3 Overall

### 3.1 Money saved per month

**What to read:** one bar per full month; above 0 the pick was cheaper than SMARD that month, a grey bar below 0 means SMARD was.

In [ ]:
PICK = {category: (model, SPLIT) for category, model in PICKS.items()}
SMARD_COLOR = STYLE["smard"]["color"]
FIGURE_LEGEND = {"loc": "upper center", "bbox_to_anchor": (0.5, 0), "frameon": False, "fontsize": 10}

# Full months: first and last day inside the test window (spec 09)
FULL_MONTHS = pd.PeriodIndex(
    [
        month
        for month in TEST_DAYS.to_period("M").unique()
        if month.start_time in TEST_DAYS and month.end_time.normalize() in TEST_DAYS
    ]
)


def pick_color(category):
    """Registry colour of a category's pick."""
    return STYLE[PICK[category][0]]["color"]


def gain_colors(values, category):
    """Pick colour where the pick saved, SMARD grey where it lost."""
    return np.where(np.asarray(values) >= 0, pick_color(category), SMARD_COLOR)


def gain_legend(fig, category, ncol=2):
    """Shared legend under the figure: who was cheaper."""
    handles = [
        Patch(color=pick_color(category), label=f"{label_of(PICK[category])} cheaper (saved)"),
        Patch(color=SMARD_COLOR, label=f"{label_of(SMARD_ROW)} cheaper (lost)"),
    ]
    fig.legend(handles=handles, ncol=ncol, **FIGURE_LEGEND)


monthly = SAVED[PICK["overall"]].groupby(COMMON.to_period("M")).sum().reindex(FULL_MONTHS) / 1e6
total = SAVED[PICK["overall"]].sum() / 1e6

fig, ax = plt.subplots(figsize=(14, 5.5), layout="constrained")
positions = np.arange(len(monthly))
ax.bar(positions, monthly.to_numpy(), width=0.7, color=gain_colors(monthly, "overall"))
ax.axhline(0, color="0.6", linewidth=0.8)
for x, value in zip(positions, monthly):
    ax.annotate(f"{value:+,.1f}", (x, value), xytext=(0, 4 if value >= 0 else -4), textcoords="offset points",
                ha="center", va="bottom" if value >= 0 else "top", fontsize=10, color="#1C1C1C")
ax.set_xticks(positions, [month.strftime("%b %Y") for month in FULL_MONTHS])
ax.margins(y=0.12)
style_plain(
    ax,
    f"Money saved vs SMARD per month, overall pick {label_of(PICK['overall'])}\n"
    f"{monthly.mean():+,.1f} M€ per full month on average, {total:+,.1f} M€ over the test year",
    "saved vs SMARD (M€)",
    "",
)
ax.yaxis.set_major_formatter(lambda value, _: f"{value:+,.0f}" if value else "0")
gain_legend(fig, "overall")
plt.show()

### 3.2 Cumulative money saved

**What to read:** a steady slope means the money comes evenly, steps mean single hours carry it; dots mark the `TOP_HOURS` hours with the largest saving or loss.

In [ ]:
def month_ticks(ax):
    """One labelled tick per month."""
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))


def hour_label(hour):
    """'Sat 05 Apr 13:00 · 1,234 EUR/MWh'."""
    return f"{hour:%a %d %b %H:%M} · {PRICE_ABS[hour]:,.0f} EUR/MWh"


saved_h = SAVED[PICK["overall"]] / 1e6
cumulative = saved_h.cumsum()
top = saved_h.abs().nlargest(TOP_HOURS).index
top_share = saved_h[top].abs().sum() / saved_h.abs().sum()

fig, ax = plt.subplots(figsize=(14, 5.5), layout="constrained")
ax.axhline(0, linewidth=1.6, color=SMARD_COLOR, linestyle=STYLE["smard"]["linestyle"])
ax.annotate(f"{label_of(SMARD_ROW)} (0)", (1, 0), xycoords=("axes fraction", "data"), xytext=(-4, 4),
            textcoords="offset points", ha="right", va="bottom", fontsize=10, color=SMARD_COLOR)
ax.plot(cumulative.index, cumulative.to_numpy(), linewidth=1.8, color=pick_color("overall"),
        label=f"{label_of(PICK['overall'])}: {total:+,.1f} M€")
ax.scatter(top, cumulative[top], s=40, color=gain_colors(saved_h[top], "overall"), edgecolor="white", linewidth=0.8,
           zorder=3, label=f"the {TOP_HOURS} hours with the largest |saved| ({100 * top_share:.0f} % of all |saved|; grey: lost)")
late = COMMON[int(0.8 * len(COMMON))]  # labels of late hours go to the left
for rank, hour in enumerate(top[:3]):
    side = -1 if hour > late else 1
    ax.annotate(
        f"{hour_label(hour)}\n{saved_h[hour]:+,.2f} M€",
        (hour, cumulative[hour]),
        xytext=(18 * side, -34 - 30 * rank),
        textcoords="offset points",
        ha="left" if side > 0 else "right",
        va="top",
        fontsize=9,
        color="#1C1C1C",
        arrowprops={"arrowstyle": "-", "color": "0.6", "linewidth": 0.8},
    )
style_timeseries(
    ax,
    f"Cumulative money saved vs SMARD, overall pick {label_of(PICK['overall'])}, {len(COMMON):,} common hours",
    "cumulative saved vs SMARD (M€)",
)
month_ticks(ax)
fig.legend(ncol=2, **FIGURE_LEGEND)
plt.show()

### 3.3 Where the money comes from

**What to read:** bar height is the saving in each `|reBAP|` band; the tile gives the band's hours and share of hours, the bar label its share of the saving. A band that holds 1 % of the hours but a large share of the saving means price spikes decide it.

In [ ]:
TILE = {"boxstyle": "round,pad=0,rounding_size=0.06", "facecolor": "#F5F7FA", "edgecolor": "#CBD2DA", "linewidth": 0.6, "alpha": 0.9}


def count_tile(ax, x, y, number, word, transform):
    """Big bold number over a small word, on a light tile centred at (x, y) in `transform` coordinates."""
    width, height, centre = max(26, 11 * len(number) + 14), 28, 16  # points
    at_spot = ax.figure.dpi_scale_trans + ScaledTranslation(x, y, transform)
    ax.add_patch(FancyBboxPatch((-width / 144, (centre - height / 2) / 72), width / 72, height / 72,
                                transform=at_spot, zorder=4, clip_on=False, **TILE))
    ax.annotate(number, (x, y), xycoords=transform, xytext=(0, centre - 0.36 * 18), textcoords="offset points",
                ha="center", va="baseline", fontsize=18, fontweight="bold", color="#1C1C1C", zorder=5, annotation_clip=False)
    ax.annotate(word, (x, y), xycoords=transform, xytext=(0, -2), textcoords="offset points",
                ha="center", va="top", fontsize=9, color="#48505A", annotation_clip=False)


def signed_ticks(axis, span):
    """Signed tick labels with as many decimals as the axis span (M€) needs."""
    decimals = 0 if span >= 10 else 1 if span >= 1 else 2
    axis.set_major_formatter(lambda value, _: f"{value:+,.{decimals}f}" if value else "0")


def plain_zero(text):
    """A label that rounds to zero loses its sign: '-0.0' -> '0.0'."""
    return text.lstrip("+-") if float(text.replace(",", "")) == 0 else text


def price_bands(category):
    """Saving per price band for a category's pick, with hours and shares."""
    in_category = HOURS_OF[category]
    if not in_category.any():
        print(f"no hour with the actual {BIN_RULE[category]} in the test window")
        return
    row = PICK[category]
    saved = SAVED.loc[in_category, row].groupby(BAND[in_category]).sum().reindex(range(len(BAND_NAMES)))
    hours = BAND[in_category].value_counts().reindex(range(len(BAND_NAMES)), fill_value=0)
    total = saved.sum()

    fig, ax = plt.subplots(figsize=(11, 5.5), layout="constrained")
    positions = np.arange(len(BAND_NAMES))
    drawn = hours > 0
    values = saved.fillna(0).to_numpy() / 1e6
    ax.bar(positions[drawn], values[drawn], width=0.6, color=gain_colors(values[drawn], category))
    ax.axhline(0, color="0.6", linewidth=0.8)
    for x, value, count in zip(positions, values, hours):
        if count:
            share = f"{plain_zero(f'{100 * saved.iloc[x] / total:.0f}')} % of saved" if total else ""
            ax.annotate(f"{plain_zero(f'{value:+,.1f}')} M€\n{share}", (x, value), xytext=(0, 4 if value >= 0 else -4),
                        textcoords="offset points", ha="center", va="bottom" if value >= 0 else "top", fontsize=10, color="#1C1C1C")
        count_tile(ax, x, 0.86, f"{count:,} h", f"{100 * count / in_category.sum():.0f} % of hours", ax.get_xaxis_transform())

    low, high = min(0, values.min()), max(0, values.max())
    ax.set_ylim(low - 0.15 * (high - low), high + 0.55 * (high - low))
    style_plain(
        ax,
        f"Where the money comes from, {category}: saving of {label_of(row)} per |reBAP| band ({in_category.sum():,} h, {total / 1e6:+,.1f} M€)",
        "saved vs SMARD (M€)",
        "",
    )
    ax.tick_params(axis="x", pad=8)

    # Band chips on the x axis: the price ramp, cheapest to dearest (after style_plain, which resets tick colours)
    ax.set_xticks(positions, [f"{name}\n{span} EUR/MWh" for name, span in zip(BAND_NAMES, BAND_RANGE)])
    for index, (tick, color) in enumerate(zip(ax.get_xticklabels(), PRICE_COLOR)):
        dark = index >= len(PRICE_COLOR) / 2
        tick.set_bbox({"boxstyle": "round,pad=0.35", "facecolor": color, "edgecolor": "none"})
        tick.set_color("white" if dark else "#1C1C1C")
    signed_ticks(ax.yaxis, high - low)
    gain_legend(fig, category)
    plt.show()


price_bands("overall")

## 4 Below zero

### 4.1 Price against advantage

**What to read:** one dot per hour with the actual below 0 MWh; above the zero line the pick was closer than SMARD, dot area is the money of that hour. Dots right of the dashed line sit in the top price band.

In [ ]:
def price_scatter(category):
    """Hourly |reBAP| against the pick's MWh advantage over SMARD, on a tail's hours."""
    in_category = HOURS_OF[category]
    if not in_category.any():
        print(f"no hour with the actual {BIN_RULE[category]} in the test window")
        return
    row = PICK[category]
    price = PRICE_ABS[in_category]
    advantage = ERRORS.loc[in_category, SMARD_ROW].abs() - ERRORS.loc[in_category, row].abs()
    saved = SAVED.loc[in_category, row]
    closer = advantage > 0

    fig, ax = plt.subplots(figsize=(12, 6), layout="constrained")
    size = 10 + 600 * saved.abs() / saved.abs().max()
    ax.scatter(price, advantage, s=size, c=gain_colors(advantage, category), alpha=0.6, edgecolor="white", linewidth=0.5)
    ax.axhline(0, color="0.6", linewidth=0.8)

    # Top-band edge, labelled directly
    edge = BAND_EDGES[-1]
    ax.axvline(edge, color=PRICE_COLOR[-1], linestyle="--", linewidth=1.2)
    ax.annotate(f"{BAND_NAMES[-1]} band: > {edge:,.0f} EUR/MWh", (edge, 1), xycoords=("data", "axes fraction"),
                xytext=(4, -4), textcoords="offset points", ha="left", va="top", fontsize=10, color=PRICE_COLOR[-1])

    # Symmetric y range: each tile sits in its own half
    reach = 1.15 * advantage.abs().max()
    ax.set_ylim(-reach, reach)
    ax.set_xscale("symlog", linthresh=10)
    ax.set_xlim(0, 2.5 * max(price.max(), edge))
    style_plain(
        ax,
        f"Price against advantage, {category} (actual {BIN_RULE[category]}): {label_of(row)} vs SMARD, {in_category.sum():,} h",
        "|e SMARD| − |e pick| (MWh)",
        "hourly mean |reBAP| (EUR/MWh, symlog)",
    )
    ax.xaxis.set_major_formatter(lambda value, _: f"{value:,.0f}")
    ax.yaxis.set_major_formatter(lambda value, _: f"{value:+,.0f}" if value else "0")

    for mask, word, y in [(closer, "pick closer", 0.86), (~closer, "SMARD closer", 0.1)]:
        count_tile(ax, 0.1, y, f"{mask.sum():,} h", f"{word}, {saved[mask].sum() / 1e6:+,.1f} M€", ax.transAxes)

    handles = [
        Line2D([], [], marker="o", linestyle="", markersize=8, color=pick_color(category), label=f"{label_of(row)} closer"),
        Line2D([], [], marker="o", linestyle="", markersize=8, color=SMARD_COLOR, label=f"{label_of(SMARD_ROW)} closer"),
    ]
    fig.legend(handles=handles, ncol=2, title="dot area: money of the hour (|saved|)", title_fontsize=10, **FIGURE_LEGEND)
    plt.show()


price_scatter("below_zero")

### 4.2 Where the money comes from

**What to read:** as in §3.3, on the hours with the actual below 0 MWh; the tiles show how many of them fall in each price band.

In [ ]:
price_bands("below_zero")

### 4.3 The decisive hours

**What to read:** the `TOP_HOURS` hours with the largest saving or loss, each with its price and both errors (`forecast − actual`, MWh); the title gives their share of all |saved| in the category.

In [ ]:
def top_hours(category):
    """The TOP_HOURS hours of a tail with the largest |saved|, as horizontal bars."""
    in_category = HOURS_OF[category]
    if not in_category.any():
        print(f"no hour with the actual {BIN_RULE[category]} in the test window")
        return
    row = PICK[category]
    saved = SAVED.loc[in_category, row] / 1e6
    top = saved.abs().nlargest(TOP_HOURS).index[::-1]  # largest on top
    share = saved[top].abs().sum() / saved.abs().sum()

    fig, ax = plt.subplots(figsize=(12, 0.45 * len(top) + 1.8), layout="constrained")
    positions = np.arange(len(top))
    ax.barh(positions, saved[top].to_numpy(), height=0.65, color=gain_colors(saved[top], category))
    ax.axvline(0, color="0.6", linewidth=0.8)
    for y, value in zip(positions, saved[top]):
        ax.annotate(f"{value:+,.2f} M€", (value, y), xytext=(4 if value >= 0 else -4, 0), textcoords="offset points",
                    ha="left" if value >= 0 else "right", va="center", fontsize=9, color="#1C1C1C")
    # Room for the value labels; no empty negative side when every hour saved
    reach, lowest = saved[top].abs().max(), saved[top].min()
    ax.set_xlim(lowest - 0.35 * reach if lowest < 0 else 0, max(0, saved[top].max()) + 0.35 * reach)
    style_plain(
        ax,
        f"The {len(top)} decisive hours, {category}: {label_of(row)} vs SMARD\n"
        f"{100 * share:.0f} % of all |saved| in the category's {in_category.sum():,} h",
        "",
        "saved vs SMARD (M€)",
    )
    ax.grid(False, axis="y")
    ax.grid(axis="x", color="0.9", linewidth=0.8)
    signed_ticks(ax.xaxis, reach)
    labels = [
        f"{hour_label(hour)} · SMARD {ERRORS.loc[hour, SMARD_ROW]:+,.0f} / pick {ERRORS.loc[hour, row]:+,.0f} MWh"
        for hour in top
    ]
    ax.set_yticks(positions, labels, fontsize=9.5)
    gain_legend(fig, category)
    plt.show()


top_hours("below_zero")

## 5 Low extreme

### 5.1 Price against advantage

**What to read:** as in §4.1, on the hours with the actual at or below P1.

In [ ]:
price_scatter("low_extreme")

### 5.2 Where the money comes from

**What to read:** as in §3.3, on the hours with the actual at or below P1.

In [ ]:
price_bands("low_extreme")

### 5.3 The decisive hours

**What to read:** as in §4.3, on the hours with the actual at or below P1.

In [ ]:
top_hours("low_extreme")

## 6 High extreme

### 6.1 Price against advantage

**What to read:** as in §4.1, on the hours with the actual above P99.

In [ ]:
price_scatter("high_extreme")

### 6.2 Where the money comes from

**What to read:** as in §3.3, on the hours with the actual above P99.

In [ ]:
price_bands("high_extreme")

### 6.3 The decisive hours

**What to read:** as in §4.3, on the hours with the actual above P99.

In [ ]:
top_hours("high_extreme")

## 7 Export

### 7.1 Cost file

`data/models/model_rebap_cost_hourly.csv`, long format: one row per common hour and row (SMARD plus every money-table row); `saved_eur` is empty on SMARD's rows. Written only with `EXPORT_ENABLED`.

In [ ]:
EXPORT_PATH = DATA_DIR / "models" / "model_rebap_cost_hourly.csv"
STAMP = "%Y-%m-%d %H:%M:%S"

frames = []
for row in [SMARD_ROW, *MODEL_ROWS]:
    forecast = smard.loc[COMMON, "fc_residual_load"] if row == SMARD_ROW else FORECAST.loc[COMMON, row]
    frames.append(
        pd.DataFrame(
            {
                "timestamp": COMMON,
                "model": row[0],
                "split_method": row[1],
                "residual_load": ACTUAL[COMMON].to_numpy(),
                "forecast": forecast.to_numpy(),
                "err_residual_load": ERRORS[row].to_numpy(),
                "rebap_mean": PRICE.to_numpy(),
                "rebap_abs_mean": PRICE_ABS.to_numpy(),
                "cost_eur": COST[row].to_numpy(),
                "saved_eur": np.nan if row == SMARD_ROW else SAVED[row].to_numpy(),
                **{f"in_{name}": IN_BIN[name].to_numpy() for name in TAILS},
            }
        )
    )
EXPORT = pd.concat(frames, ignore_index=True)

if EXPORT_ENABLED:
    EXPORT.to_csv(EXPORT_PATH, index=False, date_format=STAMP)
    print(f"wrote {EXPORT_PATH} ({len(EXPORT):,} rows: {len(COMMON):,} hours × {len(frames)} rows)")
else:
    print(f"EXPORT_ENABLED is False: nothing written ({len(EXPORT):,} rows built)")

### 7.2 Round-trip check

The file read back must hold every common hour and row once, reproduce the picks table's savings (1 EUR) and match a rebuild of each cost from the quarter-hourly prices (`1e-6` relative).

In [ ]:
def round_trip(ok, problem):
    """Stop the notebook: the written file does not match the frame."""
    if not ok:
        raise RuntimeError(f"round-trip check failed: {problem}. Re-run this notebook from a fresh kernel.")


if not EXPORT_ENABLED:
    print("EXPORT_ENABLED is False: round-trip check skipped")
else:
    back = pd.read_csv(EXPORT_PATH)
    back["timestamp"] = pd.to_datetime(back["timestamp"], format=STAMP)

    # Rows: each common hour × row exactly once
    pairs = set(zip(back["timestamp"], back["model"], back["split_method"]))
    expected = {(hour, *row) for hour in COMMON for row in [SMARD_ROW, *MODEL_ROWS]}
    round_trip(len(back) == len(expected) and pairs == expected, f"{len(back):,} rows, {len(expected):,} expected")

    # Savings per category reproduce the picks table
    for category, row in PICK.items():
        mine = back["model"].eq(row[0]) & back["split_method"].eq(row[1])
        if category != "overall":
            mine &= back[f"in_{category}"]
        gap = abs(back.loc[mine, "saved_eur"].sum() - PICKS_TABLE.loc[category, "saved (M€)"] * 1e6)
        round_trip(gap <= 1, f"{category}: saved_eur differs from the picks table by {gap:,.2f} EUR")

    # Quarter-hour rebuild: |e_h| / 4 on each quarter, times its |reBAP|, summed per hour
    on_quarters = back.merge(quarters[["hour", "reBAP unterdeckt"]], left_on="timestamp", right_on="hour")
    on_quarters["cost_q"] = on_quarters["err_residual_load"].abs() / 4 * on_quarters["reBAP unterdeckt"].abs()
    rebuilt = on_quarters.groupby(["timestamp", "model", "split_method"])["cost_q"].sum()
    written = back.set_index(["timestamp", "model", "split_method"])["cost_eur"].reindex(rebuilt.index)
    round_trip(len(rebuilt) == len(back), f"{len(back) - len(rebuilt):,} rows without quarter-hours")
    round_trip(np.allclose(rebuilt, written, rtol=1e-6, atol=0), "cost_eur differs from the quarter-hour rebuild")

    print(
        f"round-trip check passed: {len(back):,} rows, savings match the picks table, "
        f"{len(on_quarters):,} quarter-hour costs rebuild cost_eur"
    )